# Topic 3: Adaptive Query Execution (AQE) — Advanced

> **Phase 4 → Week 7 → Topic 3**
>
> Prerequisites: Topic 2 (Data Skew)

---

## What is AQE?

Before AQE, Spark compiled a query plan **once** before execution — using statistics estimates that were often wrong. AQE (Adaptive Query Execution, Spark 3.0+) **re-optimizes the plan at runtime** using actual data statistics collected from completed shuffle stages.

```
WITHOUT AQE (Spark 2.x style):
  1. Catalyst builds plan using table statistics (or guesses)
  2. Plan is fixed — executed as-is even if estimates were wrong
  3. shuffle.partitions=200 always, even for 10MB of data

WITH AQE (Spark 3.0+):
  1. Catalyst builds initial plan
  2. Stage 1 executes → shuffle write completes
                    ↓
  3. AQE re-checks plan using REAL partition sizes
  4. Re-optimizes Stage 2: coalesces tiny partitions,
     changes join strategy, splits skewed partitions
  5. Stage 2 executes with better plan
```

---

## Three AQE Features

```
1. COALESCING SHUFFLE PARTITIONS
   Problem: shuffle.partitions=200, but data is only 50MB
            → 200 tiny tasks, each taking <1ms, driver overhead dominates
   Fix: AQE merges adjacent small partitions into fewer larger ones
   Config: spark.sql.adaptive.coalescePartitions.enabled=true

2. CONVERTING SORT-MERGE JOIN → BROADCAST HASH JOIN
   Problem: Catalyst estimated one side was large (no stats)
            but at runtime it turns out to be tiny
   Fix: AQE sees actual shuffle write size, switches to BHJ
   Config: spark.sql.adaptive.localShuffleReader.enabled=true

3. SKEW JOIN HANDLING
   Problem: One partition is 100x larger than others (skew)
   Fix: AQE splits skewed partition into sub-tasks
   Config: spark.sql.adaptive.skewJoin.enabled=true
```

---

## Interview Cheat Sheet

**Q: What is AQE in Spark 3.0+ and what problems does it solve?**
> AQE re-optimizes the query plan at runtime after each shuffle stage, using actual partition statistics instead of compile-time estimates. It solves three problems: (1) too many tiny shuffle partitions (coalesces them automatically), (2) join strategy mismatch (switches SMJ to BHJ when one side turns out small), and (3) data skew in joins (splits skewed partitions into sub-tasks). Enabled by default in Spark 3.2+.

**Q: What is partition coalescing in AQE?**
> When the shuffle produces many tiny partitions (e.g., `shuffle.partitions=200` but only 50MB total data → 200 × 0.25MB partitions), AQE merges adjacent partitions until they reach `advisoryPartitionSizeInBytes` (default 64MB). This reduces task overhead and improves throughput. You set `shuffle.partitions` as a maximum, not an exact target.

**Q: Can AQE switch a join from SMJ to BHJ?**
> Yes. AQE reads the actual shuffle write size after the map stage. If one side is smaller than `autoBroadcastJoinThreshold` (default 10MB), AQE switches the join to BroadcastHashJoin. This is called a "dynamic join strategy switch" and is only possible when the table stats were inaccurate at plan time (e.g., no ANALYZE TABLE or highly filtered data).

**Q: Does enabling AQE always help?**
> AQE adds a small overhead (re-planning between stages, collecting statistics). For tiny jobs with 1-2 stages, the overhead may exceed the benefit. For complex multi-stage jobs with shuffles, AQE almost always helps. The default in Spark 3.2+ is `enabled=true` — leave it on unless you have a specific reason not to.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = SparkSession.builder \
    .appName("Week7 - AQE Advanced") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready — AQE starts disabled; we'll enable features one by one")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/16 08:08:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ready — AQE starts disabled; we'll enable features one by one


## Part 1: Coalescing Shuffle Partitions

In [2]:
# Small dataset + shuffle.partitions=200 = the "too many tiny files" problem
small_df = spark.range(10_000) \
    .withColumn("key", (F.col("id") % 50).cast("string")) \
    .withColumn("value", F.rand() * 100)

# WITHOUT AQE — always creates 200 shuffle partitions regardless of data size
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "200")

t0 = time.time()
result_no_aqe = small_df.groupBy("key").agg(F.sum("value").alias("total"))
count_no_aqe = result_no_aqe.count()
t_no_aqe = time.time() - t0

# Check actual partition count in the result
partitions_no_aqe = result_no_aqe.rdd.getNumPartitions()
print(f"Without AQE: {t_no_aqe:.3f}s, result has {partitions_no_aqe} partitions")
print(f"  → Most partitions are empty or near-empty (tiny tasks, wasted scheduling overhead)")

Without AQE: 6.415s, result has 200 partitions
  → Most partitions are empty or near-empty (tiny tasks, wasted scheduling overhead)


In [3]:
# WITH AQE partition coalescing — merges tiny partitions automatically
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "1mb")  # target per partition
spark.conf.set("spark.sql.adaptive.coalescePartitions.minPartitionSize", "128kb")

t0 = time.time()
result_aqe = small_df.groupBy("key").agg(F.sum("value").alias("total"))
result_aqe.count()
t_aqe = time.time() - t0

partitions_aqe = result_aqe.rdd.getNumPartitions()
print(f"With AQE:    {t_aqe:.3f}s, result has {partitions_aqe} partitions")
print(f"  → AQE coalesced 200 partitions into ~{partitions_aqe} based on actual data size")
print()
print("Key config:")
print("  advisoryPartitionSizeInBytes: target size for coalesced partitions (default 64MB)")
print("  minPartitionSize: floor — never coalesce below this (default 1MB in 3.2+)")
print("  shuffle.partitions is now a MAXIMUM, not an exact target")

With AQE:    0.574s, result has 1 partitions
  → AQE coalesced 200 partitions into ~1 based on actual data size

Key config:
  advisoryPartitionSizeInBytes: target size for coalesced partitions (default 64MB)
  minPartitionSize: floor — never coalesce below this (default 1MB in 3.2+)
  shuffle.partitions is now a MAXIMUM, not an exact target


In [4]:
# Visual: the coalescing algorithm
print("""
How Partition Coalescing Works:
════════════════════════════════════════════════════════════════
After shuffle write, AQE knows each partition's actual size:

Partition: [0]  [1]  [2]  [3]  [4]  [5]  [6]  [7] ... [199]
Size (KB):   2    3    0    1    4    0    2    3  ...    2
Advisory target: 1MB (1024KB)

AQE groups adjacent partitions until they reach the target:
  Group 1: [0..150]  → ~300KB total → still merge
  (keeps merging adjacently until 1MB is reached)

Result: 200 shuffle partitions → 3-5 coalesced partitions

Important: ADJACENT only — AQE won't reorder partitions.
This preserves sort order where required (e.g., range partitioning).
════════════════════════════════════════════════════════════════
""")


How Partition Coalescing Works:
════════════════════════════════════════════════════════════════
After shuffle write, AQE knows each partition's actual size:

Partition: [0]  [1]  [2]  [3]  [4]  [5]  [6]  [7] ... [199]
Size (KB):   2    3    0    1    4    0    2    3  ...    2
Advisory target: 1MB (1024KB)

AQE groups adjacent partitions until they reach the target:
  Group 1: [0..150]  → ~300KB total → still merge
  (keeps merging adjacently until 1MB is reached)

Result: 200 shuffle partitions → 3-5 coalesced partitions

Important: ADJACENT only — AQE won't reorder partitions.
This preserves sort order where required (e.g., range partitioning).
════════════════════════════════════════════════════════════════



## Part 2: Dynamic Join Strategy Switching

In [5]:
# AQE can switch a planned SortMergeJoin to BroadcastHashJoin
# at runtime when one side turns out to be small

# Simulate: a table that's heavily filtered at runtime
orders = spark.range(500_000) \
    .withColumn("order_id", F.col("id")) \
    .withColumn("customer_id", (F.col("id") % 1000).cast("string")) \
    .withColumn("amount", F.rand() * 500)

# Right side looks large initially, but after filtering only 10 rows remain
customers = spark.range(1000) \
    .withColumn("customer_id", F.col("id").cast("string")) \
    .withColumn("name", F.concat(F.lit("Customer_"), F.col("id")))

# Without broadcast hint, Catalyst may plan this as SortMergeJoin
# With AQE, if one side shuffles to tiny data, it switches to BHJ

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10mb")

join_result = orders.join(
    customers.filter(F.col("id") < 10),  # only 10 customers after filter
    orders.customer_id == customers.customer_id
)

print("Join plan (AQE may switch to BroadcastHashJoin at runtime):")
join_result.explain()
join_result.count()

Join plan (AQE may switch to BroadcastHashJoin at runtime):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [customer_id#48], [customer_id#59], Inner, BuildRight, false
   :- Filter isnotnull(customer_id#48)
   :  +- Project [id#43L, id#43L AS order_id#45L, cast((id#43L % 1000) as string) AS customer_id#48, (rand(-2693909035042452094) * 500.0) AS amount#52]
   :     +- Range (0, 500000, step=1, splits=4)
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[1, string, false]),false), [plan_id=232]
      +- Project [id#57L, cast(id#57L as string) AS customer_id#59, concat(Customer_, cast(id#57L as string)) AS name#62]
         +- Filter (id#57L < 10)
            +- Range (0, 1000, step=1, splits=4)




5000

In [6]:
print("""
AQE Join Strategy Switch:
════════════════════════════════════════════════════════════════
Scenario: stats say table B = 500MB → plan uses SortMergeJoin
Reality:  predicate pushdown filters B to 5MB

WITHOUT AQE:
  SMJ executes: both sides shuffle (expensive), then sort-merge

WITH AQE:
  Step 1: map stage (scan + filter) runs
  Step 2: AQE sees B's shuffle write = 5MB < autoBroadcastJoinThreshold
  Step 3: AQE rewrites plan → BroadcastHashJoin
  Step 4: B is broadcast, join completes without full SMJ shuffle

Config that controls this:
  spark.sql.adaptive.localShuffleReader.enabled = true  (default true)
  spark.sql.autoBroadcastJoinThreshold          = 10mb  (threshold to trigger BHJ)

When this DOESN'T trigger:
  - Both sides are large (both > threshold after filtering)
  - One side is already broadcast (no switch needed)
  - spark.sql.adaptive.enabled = false
════════════════════════════════════════════════════════════════
""")


AQE Join Strategy Switch:
════════════════════════════════════════════════════════════════
Scenario: stats say table B = 500MB → plan uses SortMergeJoin
Reality:  predicate pushdown filters B to 5MB

WITHOUT AQE:
  SMJ executes: both sides shuffle (expensive), then sort-merge

WITH AQE:
  Step 1: map stage (scan + filter) runs
  Step 2: AQE sees B's shuffle write = 5MB < autoBroadcastJoinThreshold
  Step 3: AQE rewrites plan → BroadcastHashJoin
  Step 4: B is broadcast, join completes without full SMJ shuffle

Config that controls this:
  spark.sql.adaptive.localShuffleReader.enabled = true  (default true)
  spark.sql.autoBroadcastJoinThreshold          = 10mb  (threshold to trigger BHJ)

When this DOESN'T trigger:
  - Both sides are large (both > threshold after filtering)
  - One side is already broadcast (no switch needed)
  - spark.sql.adaptive.enabled = false
════════════════════════════════════════════════════════════════



## Part 3: AQE Skew Join (Recap + Config Tuning)

In [7]:
# AQE skew join — fine-tuning the thresholds
print("""
AQE Skew Join — Config Knobs:
════════════════════════════════════════════════════════════════
spark.sql.adaptive.skewJoin.enabled                    (default: true)
  Master switch for AQE skew join.

spark.sql.adaptive.skewJoin.skewedPartitionFactor      (default: 5)
  A partition is skewed if its size >
  skewedPartitionFactor × median_partition_size
  Lower = more aggressive splitting (but may split non-skewed partitions)
  Higher = only fixes extreme skew

spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes  (default: 256MB)
  A partition is skewed only if it's ALSO larger than this threshold.
  Prevents splitting tiny partitions even if they're technically "skewed".
  For small datasets: lower this (e.g., 10MB) to see AQE in action.

How the split works:
  Skewed partition N (say 1GB) gets split into sub-tasks:
  N_0 (0-256MB), N_1 (256-512MB), N_2 (512-768MB), N_3 (768-1024MB)
  Each sub-task is joined with the FULL corresponding partition from
  the other side (that partition is read multiple times — acceptable cost).
════════════════════════════════════════════════════════════════
""")

# Configure and run
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "3")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "512kb")

print("AQE skew join configured with aggressive thresholds for demo data")


AQE Skew Join — Config Knobs:
════════════════════════════════════════════════════════════════
spark.sql.adaptive.skewJoin.enabled                    (default: true)
  Master switch for AQE skew join.

spark.sql.adaptive.skewJoin.skewedPartitionFactor      (default: 5)
  A partition is skewed if its size >
  skewedPartitionFactor × median_partition_size
  Lower = more aggressive splitting (but may split non-skewed partitions)
  Higher = only fixes extreme skew

spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes  (default: 256MB)
  A partition is skewed only if it's ALSO larger than this threshold.
  Prevents splitting tiny partitions even if they're technically "skewed".
  For small datasets: lower this (e.g., 10MB) to see AQE in action.

How the split works:
  Skewed partition N (say 1GB) gets split into sub-tasks:
  N_0 (0-256MB), N_1 (256-512MB), N_2 (512-768MB), N_3 (768-1024MB)
  Each sub-task is joined with the FULL corresponding partition from
  the other side (that pa

## Part 4: Full AQE Configuration Reference

In [8]:
print("""
Complete AQE Configuration Reference:
═══════════════════════════════════════════════════════════════════
CONFIG KEY                                          DEFAULT    NOTES
──────────────────────────────────────────────────  ─────────  ──────────────────────────────────────
spark.sql.adaptive.enabled                          true*      Master switch (* default true in 3.2+)

# Partition Coalescing
spark.sql.adaptive.coalescePartitions.enabled       true       Merge tiny post-shuffle partitions
spark.sql.adaptive.advisoryPartitionSizeInBytes     64MB       Target partition size after coalescing
spark.sql.adaptive.coalescePartitions.minPartitionSize  1MB    Floor — never go below this
spark.sql.adaptive.coalescePartitions.initialPartitionNum  200 Initial partition count for coalescing

# Join Strategy
spark.sql.adaptive.localShuffleReader.enabled       true       Allow local reads to avoid BHJ shuffle
spark.sql.autoBroadcastJoinThreshold                10MB       Table size to auto-broadcast (AQE uses
                                                               this at runtime, not just planning time)

# Skew Join
spark.sql.adaptive.skewJoin.enabled                 true       Split skewed partitions in joins
spark.sql.adaptive.skewJoin.skewedPartitionFactor   5          size > factor × median → skewed
spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes  256MB  Size floor for skewed partitions

# Observability
spark.sql.adaptive.optimizer.excludedRules          (none)     Comma-separated rules to disable
═══════════════════════════════════════════════════════════════════

Quick Setup for Production:
  spark.sql.adaptive.enabled                        = true     (already default in 3.2+)
  spark.sql.adaptive.coalescePartitions.enabled     = true
  spark.sql.adaptive.advisoryPartitionSizeInBytes   = 128MB    (larger for big clusters)
  spark.sql.adaptive.skewJoin.enabled               = true
  spark.sql.shuffle.partitions                      = 200      (AQE will reduce as needed)
""")


Complete AQE Configuration Reference:
═══════════════════════════════════════════════════════════════════
CONFIG KEY                                          DEFAULT    NOTES
──────────────────────────────────────────────────  ─────────  ──────────────────────────────────────
spark.sql.adaptive.enabled                          true*      Master switch (* default true in 3.2+)

# Partition Coalescing
spark.sql.adaptive.coalescePartitions.enabled       true       Merge tiny post-shuffle partitions
spark.sql.adaptive.advisoryPartitionSizeInBytes     64MB       Target partition size after coalescing
spark.sql.adaptive.coalescePartitions.minPartitionSize  1MB    Floor — never go below this
spark.sql.adaptive.coalescePartitions.initialPartitionNum  200 Initial partition count for coalescing

# Join Strategy
spark.sql.adaptive.localShuffleReader.enabled       true       Allow local reads to avoid BHJ shuffle
spark.sql.autoBroadcastJoinThreshold                10MB       Table size to auto-br

## Part 5: Reading AQE in explain() Output

In [9]:
# AQE adds specific nodes to the physical plan — learn to read them
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

df = spark.range(100_000) \
    .withColumn("key", (F.col("id") % 20).cast("string")) \
    .withColumn("val", F.rand() * 100)

agg = df.groupBy("key").agg(F.sum("val").alias("total"))

print("=== explain() BEFORE execution (initial plan, AQE not triggered yet): ===")
agg.explain()

# Trigger execution — AQE re-optimizes after shuffle
agg.count()

print()
print("Key nodes to look for in AQE plans:")
print("""
  AdaptiveSparkPlan        → Top-level wrapper indicating AQE is active
  CustomShuffleReader      → Coalesced/skew-adjusted partition reader
  isFinalPlan: true/false  → Whether AQE has finalized the plan
  ShuffleQueryStage        → A shuffle stage that AQE tracks for re-optimization
  BroadcastQueryStage      → A broadcast that AQE tracks

After execution with AQE, re-calling explain() shows 'isFinalPlan: true'
and the actual physical plan that ran (with any AQE modifications).
""")

=== explain() BEFORE execution (initial plan, AQE not triggered yet): ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[key#94], functions=[sum(val#97)])
   +- Exchange hashpartitioning(key#94, 200), ENSURE_REQUIREMENTS, [plan_id=371]
      +- HashAggregate(keys=[key#94], functions=[partial_sum(val#97)])
         +- Project [cast((id#92L % 20) as string) AS key#94, (rand(7388901671899063733) * 100.0) AS val#97]
            +- Range (0, 100000, step=1, splits=4)





Key nodes to look for in AQE plans:

  AdaptiveSparkPlan        → Top-level wrapper indicating AQE is active
  CustomShuffleReader      → Coalesced/skew-adjusted partition reader
  isFinalPlan: true/false  → Whether AQE has finalized the plan
  ShuffleQueryStage        → A shuffle stage that AQE tracks for re-optimization
  BroadcastQueryStage      → A broadcast that AQE tracks

After execution with AQE, re-calling explain() shows 'isFinalPlan: true'
and the actual physical plan that ran (with any AQE modifications).



In [10]:
# AQE limitations — when it can't help
print("""
AQE Limitations:
════════════════════════════════════════════════════════════════
1. STREAMING: AQE doesn't apply to Structured Streaming
   (the plan must be static for streaming pipelines)

2. BROADCAST JOIN LIMIT: AQE can switch SMJ → BHJ at runtime
   but still subject to spark.sql.autoBroadcastJoinThreshold
   and driver memory for holding the broadcast data.

3. FIRST STAGE: AQE can only re-optimize based on completed stages.
   The very first stage (no preceding shuffle) runs with the initial plan.
   Predicate pushdown must still be correct from the start.

4. COALESCING IS ONE-WAY: AQE can merge partitions (coalesce)
   but cannot split non-skewed partitions to increase parallelism.
   If your data is larger than expected, AQE won't add more partitions.
   → Set shuffle.partitions high enough (but AQE will reduce unused ones).

5. SUBQUERIES: AQE currently doesn't re-optimize within subqueries
   in all cases (improvement ongoing in newer Spark versions).

6. CATALOG-BASED STATISTICS: AQE uses actual runtime stats.
   If you're not doing joins (just map stages), AQE's dynamic join
   switch doesn't help — pre-computed ANALYZE TABLE is still valuable.
════════════════════════════════════════════════════════════════
""")


AQE Limitations:
════════════════════════════════════════════════════════════════
1. STREAMING: AQE doesn't apply to Structured Streaming
   (the plan must be static for streaming pipelines)

2. BROADCAST JOIN LIMIT: AQE can switch SMJ → BHJ at runtime
   but still subject to spark.sql.autoBroadcastJoinThreshold
   and driver memory for holding the broadcast data.

3. FIRST STAGE: AQE can only re-optimize based on completed stages.
   The very first stage (no preceding shuffle) runs with the initial plan.
   Predicate pushdown must still be correct from the start.

4. COALESCING IS ONE-WAY: AQE can merge partitions (coalesce)
   but cannot split non-skewed partitions to increase parallelism.
   If your data is larger than expected, AQE won't add more partitions.
   → Set shuffle.partitions high enough (but AQE will reduce unused ones).

5. SUBQUERIES: AQE currently doesn't re-optimize within subqueries
   in all cases (improvement ongoing in newer Spark versions).

6. CATALOG-BASED ST

## Part 6: AQE vs Manual Optimization Tradeoffs

In [11]:
print("""
AQE vs Manual Optimization — When to Use Each:
════════════════════════════════════════════════════════════════
SITUATION                      USE AQE?    MANUAL OPTIMIZATION?
─────────────────────────────  ──────────  ─────────────────────────────────
Shuffle partitions too many    ✓ First      Only if AQE can't reduce enough
Shuffle partitions too few     ✗ Can't help Increase shuffle.partitions
Mild skew (5-20×)              ✓ AQE handles Most cases
Extreme skew (1000×)           Partial      Add salting for determinism
SMJ on small-after-filter table ✓ Dynamic    Only if AQE is disabled
Known small broadcast table    ✗ N/A        Use broadcast() hint directly
Consistent data, fixed schema  ✗ Overhead   Static plan (disable AQE for speed)
Unknown/variable data          ✓ Always     Combine: AQE + good partitioning

Rule: Enable AQE + set shuffle.partitions high (200-400).
      Let AQE auto-reduce. Only salt when AQE isn't enough for extreme skew.
════════════════════════════════════════════════════════════════
""")


AQE vs Manual Optimization — When to Use Each:
════════════════════════════════════════════════════════════════
SITUATION                      USE AQE?    MANUAL OPTIMIZATION?
─────────────────────────────  ──────────  ─────────────────────────────────
Shuffle partitions too many    ✓ First      Only if AQE can't reduce enough
Shuffle partitions too few     ✗ Can't help Increase shuffle.partitions
Mild skew (5-20×)              ✓ AQE handles Most cases
Extreme skew (1000×)           Partial      Add salting for determinism
SMJ on small-after-filter table ✓ Dynamic    Only if AQE is disabled
Known small broadcast table    ✗ N/A        Use broadcast() hint directly
Consistent data, fixed schema  ✗ Overhead   Static plan (disable AQE for speed)
Unknown/variable data          ✓ Always     Combine: AQE + good partitioning

Rule: Enable AQE + set shuffle.partitions high (200-400).
      Let AQE auto-reduce. Only salt when AQE isn't enough for extreme skew.
══════════════════════════════════

## Exercises

1. Run a `groupBy` on a small DataFrame (10,000 rows) with `shuffle.partitions=200` — first with AQE disabled, then enabled. Compare result partition counts.
2. Create a DataFrame where one key has 95% of all rows. Enable AQE skew join with `skewedPartitionFactor=3` and `skewedPartitionThresholdInBytes=100kb`. Confirm AQE splits the skewed partition.
3. Explain why AQE can switch a SortMergeJoin to BroadcastHashJoin at runtime but cannot switch it to a CrossJoin.
4. You have `shuffle.partitions=200` and AQE coalesces to 5 partitions. Your next query does a join — does it start with 200 or 5 partitions? Why?
5. What happens if you set `spark.sql.adaptive.advisoryPartitionSizeInBytes=1gb` on a 100MB dataset with 200 shuffle partitions?

In [12]:
# Exercise 1: Coalescing demo
test_df = spark.range(10_000) \
    .withColumn("key", (F.col("id") % 10).cast("string")) \
    .withColumn("val", F.rand())

spark.conf.set("spark.sql.shuffle.partitions", "200")

# Without AQE
spark.conf.set("spark.sql.adaptive.enabled", "false")
no_aqe = test_df.groupBy("key").agg(F.sum("val"))
no_aqe.count()
print(f"Without AQE: {no_aqe.rdd.getNumPartitions()} partitions in result")

# With AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "1mb")
with_aqe = test_df.groupBy("key").agg(F.sum("val"))
with_aqe.count()
print(f"With AQE:    {with_aqe.rdd.getNumPartitions()} partitions in result")
print()
print("Answer Q4: Each NEW query starts fresh with shuffle.partitions=200")
print("AQE coalescing only affects the CURRENT query's output partitions")

Without AQE: 200 partitions in result


With AQE:    1 partitions in result

Answer Q4: Each NEW query starts fresh with shuffle.partitions=200
AQE coalescing only affects the CURRENT query's output partitions
